In [1]:
import os
import json
import glob

In [2]:
def count_single_timepoint_municipalities(results_dir="resultados"):
    """
    Count and list municipalities that have only one unique time series point

    Args:
        results_dir (str): Directory containing the result JSON files

    Returns:
        tuple: (count, list_of_municipalities)
    """
    # Find all JSON result files
    result_files = glob.glob(os.path.join(results_dir, '*_results.json'))

    # Skip the processing_summary.json and consolidated_report.json if they exist
    result_files = [f for f in result_files if
                    not f.endswith('processing_summary.json') and
                    not f.endswith('consolidated_report.json')]

    single_timepoint_municipalities = []

    # Process each result file
    for result_file in result_files:
        try:
            with open(result_file, 'r', encoding='utf-8') as f:
                result = json.load(f)

            # Check if this has the specific message
            if (result.get('status') == 'warning' and
                    result.get('message') == "Not enough data: dataset contains only one unique time series point"):
                # Extract municipality name from filename
                municipio = os.path.basename(result_file).replace('_results.json', '')
                single_timepoint_municipalities.append(municipio)

        except Exception as e:
            print(f"Error reading {result_file}: {str(e)}")

    return len(single_timepoint_municipalities), single_timepoint_municipalities

In [3]:
# Run the analysis
count, municipalities = count_single_timepoint_municipalities()

In [4]:
print(f"Total municipalities with only one time point: {count}")
print(f"Percentage of single-timepoint municipalities: {count/len(glob.glob(os.path.join('resultados', '*_results.json')))*100:.2f}%")

Total municipalities with only one time point: 2
Percentage of single-timepoint municipalities: 0.18%


In [5]:
# Optionally print the list of municipalities
if count > 0:
    print("\nList of municipalities with only one time point:")
    for i, municipio in enumerate(municipalities, 1):
        print(f"{i}. {municipio}")


List of municipalities with only one time point:
1. AMAZONAS_MIRITI___PARANÁ
2. AMAZONAS_PUERTO_SANTANDER


In [9]:
import os
import pandas as pd
import glob

def create_municipality_dataset(directory_path, output_file='municipalities.csv'):
    """
    Extract department and municipality names from filenames in the directory
    and export them to a CSV file.

    Args:
        directory_path (str): Path to the directory containing municipality CSV files
        output_file (str): Output CSV file name

    Returns:
        pd.DataFrame: DataFrame containing the extracted information
    """
    # Check if the directory exists
    if not os.path.exists(directory_path):
        raise FileNotFoundError(f"Directory not found: {directory_path}")

    # Get all CSV files in the directory
    file_pattern = os.path.join(directory_path, "*.csv")
    file_paths = glob.glob(file_pattern)

    if not file_paths:
        print(f"No CSV files found in {directory_path}")
        return None

    # Extract department and municipality names from filenames
    data = []
    for file_path in file_paths:
        filename = os.path.basename(file_path)
        if '.' in filename:
            # Split filename by dot and remove the .csv extension
            parts = filename.replace('.csv', '').split('.')
            if len(parts) >= 2:
                department = parts[0]
                municipality = '.'.join(parts[1:])  # Handle cases with multiple dots
                data.append({
                    'Departamento': department,
                    'Municipio': municipality
                })

    # Create DataFrame
    df = pd.DataFrame(data)

    # Sort by department and municipality
    if not df.empty:
        df = df.sort_values(['Departamento', 'Municipio'])

    # Save to CSV
    df.to_csv(output_file, index=False)
    print(f"Dataset saved to {output_file} with {len(df)} municipalities")

    return df


In [10]:
# Example usage
if __name__ == "__main__":
    # Use relative path as specified
    directory_path = "../../Limpieza/data/subdatasets-ubicacion"

    try:
        municipality_df = create_municipality_dataset(directory_path)

        if municipality_df is not None and not municipality_df.empty:
            # Print some statistics
            print(f"\nTotal number of departments: {municipality_df['Departamento'].nunique()}")
            print(f"Top 5 departments by number of municipalities:")
            dept_counts = municipality_df['Departamento'].value_counts().head(5)
            for dept, count in dept_counts.items():
                print(f"  - {dept}: {count} municipalities")
    except Exception as e:
        print(f"Error creating municipality dataset: {str(e)}")

Dataset saved to municipalities.csv with 1121 municipalities

Total number of departments: 33
Top 5 departments by number of municipalities:
  - ANTIOQUIA: 125 municipalities
  - BOYACÁ: 123 municipalities
  - CUNDINAMARCA: 116 municipalities
  - SANTANDER: 87 municipalities
  - NARIÑO: 64 municipalities
